In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("PostgreSQL JDBC Read") \
    .config("spark.jars", "/home/user/jars/postgresql-42.7.3.jar") \
    .getOrCreate()
# Создать сеанс Spark
spark = SparkSession.builder.appName( "SimplePySparkJob" ).getOrCreate()

# DataFrame продуктов
df_prod = spark.read.csv("prod.csv", header= True , inferSchema= True )
# DataFrame категорий
df_cat = spark.read.csv("cat.csv", header= True , inferSchema= True )
# DataFrame связующая табл
df_prod_cat = spark.read.csv("prod_cat.csv", header= True , inferSchema= True )

# df_prod.show(truncate=False)
# df_cat.show(truncate=False)
# df_prod_cat.show(truncate=False)

# join категорий со связ таблицей
j_cat = df_cat.join(
    df_prod_cat,
    on="cat_id",
    how="left"
)

# j_cat.show(truncate=False)

# результат  «Имя продукта – Имя категории» и имена всех продуктов, у которых нет категорий
j_prod = df_prod.join(
    j_cat,
    on="prod_id",
    how="left"
).select(
    col("name_prod"),
    col("name_cat"))

j_prod.show(truncate=False)
spark.stop()

+---------+--------------------+
|name_prod|name_cat            |
+---------+--------------------+
|Бананы   |Овощи-фрукты        |
|Печенье  |Кондитерские изделия|
|Колбаса  |Мясная продукция    |
|Молоко   |Молочная продукция  |
|Сметана  |Молочная продукция  |
|Энергетик|NULL                |
+---------+--------------------+

